# Класифікація: LogisticRegression на Iris dataset

У цьому notebook показаний повний приклад multiclass classification: модель `LogisticRegression` розпізнає вид ірису за 4 числовими ознаками квітки.

## Мета

Побудувати повний classification workflow:

- завантажити датасет з `sklearn.datasets`;
- розділити дані на train/test;
- навчити `LogisticRegression`;
- порахувати Accuracy, Precision, Recall, F1;
- прочитати confusion matrix.

Очікуваний головний результат: accuracy приблизно `0.91`, а основні помилки мають бути між класами `versicolor` і `virginica`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Завантажуємо Iris dataset

`load_iris(as_frame=True)` повертає дані у форматі, зручному для `pandas`. Числовий `target` перетворимо на зрозумілі назви класів.

In [ ]:
data = load_iris(as_frame=True)

df = data.frame.copy()
df["species"] = df["target"].map(dict(enumerate(data.target_names)))
df = df.drop(columns="target")

print(df.shape)
df.head()

**Очікуваний результат**

- Розмір таблиці: `(150, 5)`.
- Є 4 числові ознаки: `sepal length (cm)`, `sepal width (cm)`, `petal length (cm)`, `petal width (cm)`.
- Цільова колонка: `species` з класами `setosa`, `versicolor`, `virginica`.

## 2. Швидкий огляд даних

Перевіримо описові статистики та баланс класів.

In [ ]:
display(df.describe().T)

class_counts = df["species"].value_counts().sort_index()
display(class_counts.to_frame("count"))

fig, ax = plt.subplots(figsize=(7, 4))
class_counts.plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Кількість прикладів у кожному класі")
ax.set_xlabel("species")
ax.set_ylabel("count")
plt.xticks(rotation=0)
plt.show()

**Висновок**

Кожен клас має по `50` прикладів. Це збалансований датасет, тому `accuracy` тут можна читати простіше, ніж у задачах з сильним дисбалансом класів.

## 3. Візуалізуємо дві ознаки

Подивимось, як класи розділяються за довжиною і шириною пелюстки.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for species, part in df.groupby("species"):
    ax.scatter(part["petal length (cm)"], part["petal width (cm)"], alpha=0.8, label=species)

ax.set_title("Iris: petal length vs petal width")
ax.set_xlabel("petal length (cm)")
ax.set_ylabel("petal width (cm)")
ax.legend(title="species")
plt.show()

**Висновок**

`setosa` добре відділяється від інших класів. `versicolor` і `virginica` частково ближчі один до одного, тому саме між ними модель може робити помилки.

## 4. Готуємо X, y і train/test split

`X` містить 4 числові ознаки, `y` містить назву виду ірису.

Використовуємо `stratify=y`, щоб у train і test зберегти однакову пропорцію класів.

In [ ]:
X = df.drop(columns="species")
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Test class counts:")
print(y_test.value_counts().sort_index())

**Очікуваний результат**

- `X_train.shape`: `(105, 4)`.
- `X_test.shape`: `(45, 4)`.
- У тестовій вибірці має бути по `15` прикладів кожного класу.

## 5. Створюємо і навчаємо LogisticRegression pipeline

Масштабування потрібне, бо логістична регресія чутлива до масштабу ознак.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)

## 6. Отримуємо прогнози й рахуємо метрики

Для multiclass classification використаємо `average="macro"`, щоб кожен клас мав однакову вагу в Precision, Recall і F1.

In [ ]:
y_pred = model.predict(X_test)

metrics = pd.DataFrame([
    {"metric": "accuracy", "value": accuracy_score(y_test, y_pred)},
    {"metric": "precision_macro", "value": precision_score(y_test, y_pred, average="macro")},
    {"metric": "recall_macro", "value": recall_score(y_test, y_pred, average="macro")},
    {"metric": "f1_macro", "value": f1_score(y_test, y_pred, average="macro")},
])

metrics.round(3)

**Очікуваний результат**

Метрики мають бути приблизно такими:

| metric | value |
|---|---:|
| accuracy | 0.911 |
| precision_macro | 0.916 |
| recall_macro | 0.911 |
| f1_macro | 0.911 |

## 7. Confusion matrix

Confusion matrix показує, які саме класи модель плутає між собою.

In [ ]:
labels = list(data.target_names)
cm = confusion_matrix(y_test, y_pred, labels=labels)

print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
display_cm.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix: Iris classification")
plt.show()

**Очікуваний результат**

Confusion matrix має бути близькою до такої:

```text
[[15  0  0]
 [ 0 14  1]
 [ 0  3 12]]
```

Інтерпретація:

- `setosa` класифікується без помилок;
- основні помилки між `versicolor` і `virginica`;
- це логічно, бо ці два класи на scatter plot розташовані ближче один до одного.

## 8. Classification report

`classification_report` показує метрики по кожному класу окремо.

In [ ]:
print(classification_report(y_test, y_pred, labels=labels))

**Висновок**

Для `setosa` precision, recall і F1 дорівнюють `1.00`. Для `versicolor` і `virginica` метрики нижчі, бо саме між ними є кілька помилок.

## 9. Ймовірності класів

`LogisticRegression` може повертати не тільки клас, а й ймовірності класів через `predict_proba`.

In [ ]:
proba = model.predict_proba(X_test)
proba_df = pd.DataFrame(proba, columns=model.classes_)
proba_df["true_class"] = y_test.reset_index(drop=True)
proba_df["predicted_class"] = y_pred
proba_df.head(10).round(3)

**Висновок**

У кожному рядку ймовірності трьох класів сумуються приблизно до `1.0`. Найбільша ймовірність відповідає передбаченому класу.

## Фінальний висновок

Модель добре класифікує Iris dataset: accuracy приблизно `0.91`. Клас `setosa` відділяється найкраще, а помилки виникають між `versicolor` і `virginica`. Macro-метрики доречні, бо ми хочемо оцінити якість по кожному класу, а не тільки загальну частку правильних відповідей.